In [ ]:
#  1st step: separate the lysosome volume into each file (etc. 6672_image_1_A.csv, 6672_image_1_B.csv), and will store these files into one folder

import pandas as pd
import os

# Load your CSV
file_path = "/Users/linyufen/Downloads/lysosome_2025.csv" # replace with your file location, and please use csv instead of excel
df = pd.read_csv(file_path)

# Create output folder
output_dir = "/Users/linyufen/Downloads/separated_csv" # replace with your desired location
os.makedirs(output_dir, exist_ok=True)

# Process each pair of vol/mgl columns
for col in df.columns:
    if col.endswith("_mgl"):
        base = col.replace("_mgl", "")
        vol_col = base + "_vol"
        
        if vol_col in df.columns:
            # Drop NaN first
            valid_data = df.dropna(subset=[col])
            
            # Split by each unique category in _mgl
            for category in valid_data[col].unique():
                subset = valid_data[valid_data[col] == category][[vol_col, col]]
                out_name = f"{base}_{category}.csv"
                subset.to_csv(os.path.join(output_dir, out_name), index=False)


print("✅ Files saved in 'separated_csv' folder.")


✅ Files saved in 'separated_csv' folder.


In [ ]:
#  2nd step: from the folder from previous step, we will filter 0 to 1 and 1 ro 15 into another two different folders

import pandas as pd
import os

# Input/output folders
input_dir = "/Users/linyufen/Downloads/separated_csv" # replace with folder from previous step 
output_dir = "/Users/linyufen/Downloads/filtered_0to1_csv" # replace with desired location
os.makedirs(output_dir, exist_ok=True)

# Loop through all CSV files in the separated folder
for file in os.listdir(input_dir):
    if file.endswith(".csv"):
        file_path = os.path.join(input_dir, file)
        
        # Load
        df = pd.read_csv(file_path)
        
        # Find the "vol" column (should be the first column)
        vol_col = df.columns[0]
        
        # Filter values between 0 and 1
        filtered = df[(df[vol_col] >= 0) & (df[vol_col] <= 1)]
        
        # Save to new folder
        filtered.to_csv(os.path.join(output_dir, file), index=False)

print("✅ All filtered CSVs saved in 'filtered_0to1_csv' folder.")


import pandas as pd
import os

# Input/output folders
input_dir = "/Users/linyufen/Downloads/separated_csv" # replace with folder from previous step 
output_dir = "/Users/linyufen/Downloads/filtered_1to15__csv" # replace with desired location
os.makedirs(output_dir, exist_ok=True)

# Loop through all CSV files in the separated folder
for file in os.listdir(input_dir):
    if file.endswith(".csv"):
        file_path = os.path.join(input_dir, file)
        
        # Load
        df = pd.read_csv(file_path)
        
        # Find the "vol" column (should be the first column)
        vol_col = df.columns[0]
        
        # Filter values between 1 and 15
        filtered = df[(df[vol_col] > 1) & (df[vol_col] <= 15)]
        
        # Save to new folder
        filtered.to_csv(os.path.join(output_dir, file), index=False)

print("✅ All filtered CSVs saved in 'filtered_1to15_csv' folder.")



✅ All filtered CSVs saved in 'filtered_0to1_csv' folder.
✅ All filtered CSVs saved in 'filtered_1to15_csv' folder.


In [ ]:
# 3rd step: get the patient files you want and save as another folder, now we complete the data preprocessing

import os
import shutil

# List of UWA codes
uwa_codes = ["6672", "6674", "6774", "6789", "6795", "6805", "6815", "6904", "6992", "7065"] # replace with the patient number you want to get 

# Input/output folders
input_dir = "/Users/linyufen/Downloads/filtered_0to1_csv" #replce
output_dir = "/Users/linyufen/Downloads/uwa_filtered_0to1_csv" #replace 
os.makedirs(output_dir, exist_ok=True)

# Copy only files that match UWA codes
for file in os.listdir(input_dir):
    if file.endswith(".csv"):
        # Check if filename starts with any of the UWA codes
        if any(file.startswith(code) for code in uwa_codes):
            shutil.copy(os.path.join(input_dir, file), os.path.join(output_dir, file))

print("✅ UWA files copied to 'uwa_filtered_0to1_csv' folder.")


# Input/output folders
input_dir = "/Users/linyufen/Downloads/filtered_0to1_csv"
output_dir = "/Users/linyufen/Downloads/uwa_filtered_1to15_csv"
os.makedirs(output_dir, exist_ok=True)

# Copy only files that match UWA codes
for file in os.listdir(input_dir):
    if file.endswith(".csv"):
        # Check if filename starts with any of the UWA codes
        if any(file.startswith(code) for code in uwa_codes):
            shutil.copy(os.path.join(input_dir, file), os.path.join(output_dir, file))

print("✅ UWA files copied to 'uwa_filtered_1to15_csv' folder.")

✅ UWA files copied to 'uwa_filtered_0to1_csv' folder.
✅ UWA files copied to 'uwa_filtered_1to15_csv' folder.


In [ ]:
# 4th step: now we start to do the analysis, first calculate the lysosome mean volume and the lysosome count of each image from each patient


import pandas as pd
import os

def pick_vol_column(df: pd.DataFrame) -> str:
    """Prefer an explicit *_vol column; otherwise fall back to the first column."""
    vol_cols = [c for c in df.columns if c.endswith("_vol")]
    return vol_cols[0] if vol_cols else df.columns[0]

def summarize_folder(input_dir: str, output_path: str):
    results = []
    for file in os.listdir(input_dir):
        if not file.endswith(".csv"):
            continue
        file_path = os.path.join(input_dir, file)
        df = pd.read_csv(file_path)

        # Row count (as in your original script)
        row_count = len(df)

        # Mean of volume column
        vol_col = pick_vol_column(df)
        vals = pd.to_numeric(df[vol_col], errors="coerce")
        mean_value = float(vals.mean()) if len(vals) > 0 else float("nan")

        results.append({
            "filename": file,
            "num_rows": row_count,
            "mean_value": mean_value
        })

    pd.DataFrame(results).to_csv(output_path, index=False)
    print(f"✅ Saved: {output_path}")

# 0–1 folder
summarize_folder(
    input_dir="/Users/linyufen/Downloads/uwa_filtered_0to1_csv",
    output_path="/Users/linyufen/Downloads/uwa_0to1_counts.csv"
)

# 1–15 folder
summarize_folder(
    input_dir="/Users/linyufen/Downloads/uwa_filtered_1to15_csv",
    output_path="/Users/linyufen/Downloads/uwa_1to15_counts.csv"
)


✅ Saved: /Users/linyufen/Downloads/uwa_0to1_counts.csv
✅ Saved: /Users/linyufen/Downloads/uwa_1to15_counts.csv


In [ ]:
# 5th step: then we use the count from the previous step and merge with the cell volume, and calculate the density

import os
import pandas as pd

# ---- CONFIG ----
COUNTS_PATH = "/Users/linyufen/Downloads/uwa_0to1_counts.csv"  # your summary with filename,num_rows
VOLUME_PATH = "/Users/linyufen/Downloads/mgl_2025.csv"        # the big table with *_vol and *_mgl pairs
OUTPUT_PATH = "/Users/linyufen/Downloads/uwa_0to1_counts_with_volume_density.csv"
# ----------------

def parse_filename(fname: str):
    """'6672_image_1_A.csv' -> base='6672_image_1', label='A'"""
    stem = os.path.splitext(os.path.basename(fname))[0]
    parts = stem.split("_")
    label = parts[-1]
    base = "_".join(parts[:-1])
    return base, label

def main():
    # Load inputs
    counts_df = pd.read_csv(COUNTS_PATH)
    vol_df = pd.read_csv(VOLUME_PATH)

    results = []
    for _, row in counts_df.iterrows():
        fname = row["filename"]
        num_rows = int(row["num_rows"])

        base, label = parse_filename(fname)
        vol_col = f"{base}_vol"
        mgl_col = f"{base}_mgl"

        if vol_col in vol_df.columns and mgl_col in vol_df.columns:
            mask = vol_df[mgl_col] == label
            mask = mask.fillna(False)

            # Sum volume for rows with this label
            vol_sum = vol_df.loc[mask, vol_col].sum(min_count=1)  # NaN if no match

            density = (num_rows / vol_sum) if pd.notna(vol_sum) and vol_sum != 0 else float("nan")

            results.append({
                "filename": fname,
                "uwa_code": base.split("_")[0],
                "image_id": base,
                "label": label,
                "num_rows": num_rows,
                "volume_sum": vol_sum,
                "density_rows_per_volume": density,
            })
        else:
            # Columns missing: keep a note
            miss = []
            if vol_col not in vol_df.columns: miss.append(vol_col)
            if mgl_col not in vol_df.columns: miss.append(mgl_col)
            results.append({
                "filename": fname,
                "uwa_code": base.split("_")[0],
                "image_id": base,
                "label": label,
                "num_rows": num_rows,
                "volume_sum": float("nan"),
                "density_rows_per_volume": float("nan"),
                "note": f"Missing columns: {', '.join(miss)}"
            })

    out = pd.DataFrame(results).sort_values(["uwa_code", "image_id", "label"])
    out.to_csv(OUTPUT_PATH, index=False)
    print(f"✅ Saved: {OUTPUT_PATH}")

if __name__ == "__main__":
    main()


# ---- CONFIG ----
COUNTS_PATH = "/Users/linyufen/Downloads/uwa_1to15_counts.csv"  # your summary with filename,num_rows
VOLUME_PATH = "/Users/linyufen/Downloads/mgl_2025.csv"        # the big table with *_vol and *_mgl pairs
OUTPUT_PATH = "/Users/linyufen/Downloads/uwa_1to15_counts_with_volume_density.csv"
# ----------------

def parse_filename(fname: str):
    """'6672_image_1_A.csv' -> base='6672_image_1', label='A'"""
    stem = os.path.splitext(os.path.basename(fname))[0]
    parts = stem.split("_")
    label = parts[-1]
    base = "_".join(parts[:-1])
    return base, label

def main():
    # Load inputs
    counts_df = pd.read_csv(COUNTS_PATH)
    vol_df = pd.read_csv(VOLUME_PATH)

    results = []
    for _, row in counts_df.iterrows():
        fname = row["filename"]
        num_rows = int(row["num_rows"])

        base, label = parse_filename(fname)
        vol_col = f"{base}_vol"
        mgl_col = f"{base}_mgl"

        if vol_col in vol_df.columns and mgl_col in vol_df.columns:
            mask = vol_df[mgl_col] == label
            mask = mask.fillna(False)

            # Sum volume for rows with this label
            vol_sum = vol_df.loc[mask, vol_col].sum(min_count=1)  # NaN if no match

            density = (num_rows / vol_sum) if pd.notna(vol_sum) and vol_sum != 0 else float("nan")

            results.append({
                "filename": fname,
                "uwa_code": base.split("_")[0],
                "image_id": base,
                "label": label,
                "num_rows": num_rows,
                "volume_sum": vol_sum,
                "density_rows_per_volume": density,
            })
        else:
            # Columns missing: keep a note
            miss = []
            if vol_col not in vol_df.columns: miss.append(vol_col)
            if mgl_col not in vol_df.columns: miss.append(mgl_col)
            results.append({
                "filename": fname,
                "uwa_code": base.split("_")[0],
                "image_id": base,
                "label": label,
                "num_rows": num_rows,
                "volume_sum": float("nan"),
                "density_rows_per_volume": float("nan"),
                "note": f"Missing columns: {', '.join(miss)}"
            })

    out = pd.DataFrame(results).sort_values(["uwa_code", "image_id", "label"])
    out.to_csv(OUTPUT_PATH, index=False)
    print(f"✅ Saved: {OUTPUT_PATH}")
    
if __name__ == "__main__":
    main()

✅ Saved: /Users/linyufen/Downloads/uwa_0to1_counts_with_volume_density.csv
✅ Saved: /Users/linyufen/Downloads/uwa_1to15_counts_with_volume_density.csv


In [ ]:
# 6th step: we succesfully get the density from previous step. if you want to add the demographic data of each patient in the analysis files which could make it easier to do the further anaylsis.

import os
import re
import pandas as pd

# ====== CONFIG ======
UWA_0TO1_INPUT = "/Users/linyufen/Downloads/uwa_0to1_counts_with_volume_density.csv"
UWA_1TO15_INPUT = "/Users/linyufen/Downloads/uwa_1to15_counts_with_volume_density.csv"
JAYADEV_PATH   = "/Users/linyufen/Downloads/Jayadev microglia in ctx_case cohort_20250801.csv"

OUTPUT_0TO1 = "/Users/linyufen/Downloads/uwa_0to1_with_microglia_merged.csv"
OUTPUT_1TO15 = "/Users/linyufen/Downloads/uwa_1to15_with_microglia_merged.csv"
# ====================

def extract_code(val):
    """
    Extract a 4-digit (preferred) or 4+ digit numeric code from any string.
    Examples: 'UWA-6815' -> '6815', 'Patient 6992 trial A' -> '6992'
    """
    s = str(val)
    m4 = re.search(r"\b(\d{4})\b", s)
    if m4:
        return m4.group(1)
    m = re.search(r"(\d{4,})", s)
    return m.group(1) if m else None

def ensure_uwa_code(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ensure dataframe has a string column 'uwa_code'.
    If missing, derive from 'filename' (prefix before first underscore).
    """
    out = df.copy()
    if "uwa_code" not in out.columns:
        if "filename" not in out.columns:
            raise ValueError("UWA file must contain 'uwa_code' or 'filename'.")
        out["uwa_code"] = out["filename"].astype(str).str.split("_", n=1, expand=True)[0]
    out["uwa_code"] = out["uwa_code"].astype(str).str.strip()
    return out

def detect_jayadev_code_column(jay: pd.DataFrame, uwa_codes: set) -> tuple[str, pd.Series]:
    """
    Auto-detect which Jayadev column best matches UWA codes.
    Returns (column_name, normalized_code_series).
    """
    best_col, best_hits, best_norm = None, -1, None
    for col in jay.columns:
        norm = jay[col].apply(extract_code)
        hits = int(norm.isin(uwa_codes).sum())
        if hits > best_hits:
            best_col, best_hits, best_norm = col, hits, norm
    if best_col is None or best_hits == 0:
        raise ValueError("Could not find a matching patient code column in the Jayadev file.")
    return best_col, best_norm

def prepare_jayadev(jay: pd.DataFrame, uwa_codes: set, best_norm: pd.Series) -> pd.DataFrame:
    """
    Keep only Jayadev rows that map to UWA codes. Deduplicate per code (keep first).
    """
    j = jay.copy()
    j["uwa_code"] = best_norm.astype(str).str.strip()
    j = j[j["uwa_code"].isin(uwa_codes)].copy()
    j = j.drop_duplicates(subset=["uwa_code"], keep="first")
    return j

def safe_merge(uwa_df: pd.DataFrame, jay_norm: pd.DataFrame) -> pd.DataFrame:
    """
    Left-join Jayadev patient-level data to UWA trial-level rows on 'uwa_code'.
    Colliding column names from Jayadev (except 'uwa_code') are suffixed with '_jay'.
    """
    uwa_df = ensure_uwa_code(uwa_df)
    overlap = set(uwa_df.columns).intersection(jay_norm.columns) - {"uwa_code"}
    if overlap:
        jay_norm = jay_norm.rename(columns={c: f"{c}_jay" for c in overlap})
    return uwa_df.merge(jay_norm, on="uwa_code", how="left")

def main():
    # Load UWA trial files
    if not os.path.exists(UWA_0TO1_INPUT):
        raise FileNotFoundError(UWA_0TO1_INPUT)
    if not os.path.exists(UWA_1TO15_INPUT):
        raise FileNotFoundError(UWA_1TO15_INPUT)
    if not os.path.exists(JAYADEV_PATH):
        raise FileNotFoundError(JAYADEV_PATH)

    uwa_0to1 = ensure_uwa_code(pd.read_csv(UWA_0TO1_INPUT))
    uwa_1to15 = ensure_uwa_code(pd.read_csv(UWA_1TO15_INPUT))
    jay = pd.read_csv(JAYADEV_PATH)

    # Detect Jayadev code column using union of UWA codes (more robust)
    all_codes = set(uwa_0to1["uwa_code"].unique()).union(set(uwa_1to15["uwa_code"].unique()))
    best_col, best_norm = detect_jayadev_code_column(jay, all_codes)

    # Prepare Jayadev table
    jay_norm = prepare_jayadev(jay, all_codes, best_norm)

    # Merge separately and save
    merged_0to1 = safe_merge(uwa_0to1, jay_norm)
    merged_1to15 = safe_merge(uwa_1to15, jay_norm)

    # Optional: sort nicely if columns exist
    sort_cols = [c for c in ["uwa_code", "image_id", "label", "filename"] if c in merged_0to1.columns]
    if sort_cols:
        merged_0to1 = merged_0to1.sort_values(sort_cols)
        merged_1to15 = merged_1to15.sort_values(sort_cols)

    merged_0to1.to_csv(OUTPUT_0TO1, index=False)
    merged_1to15.to_csv(OUTPUT_1TO15, index=False)

    print("✅ Merge complete")
    print(f"• Jayadev code column matched: {best_col}")
    print(f"• 0–1 rows: {len(merged_0to1)}  → saved to {OUTPUT_0TO1}")
    print(f"• 1–15 rows: {len(merged_1to15)} → saved to {OUTPUT_1TO15}")
    print(f"• Jayadev patients matched: {len(jay_norm)}")

if __name__ == "__main__":
    main()


✅ Merge complete
• Jayadev code column matched: UWA-
• 0–1 rows: 126  → saved to /Users/linyufen/Downloads/uwa_0to1_with_microglia_merged.csv
• 1–15 rows: 126 → saved to /Users/linyufen/Downloads/uwa_1to15_with_microglia_merged.csv
• Jayadev patients matched: 10


In [ ]:
# now we complete our analysis, remember to replace all the file name and location
# we first do the preprocessing to split the data in to different files for each image of each patint
# then we calculate the number of count and the mean volume and save the output as a csv
# next, we calculate the density and save the output as another csv
# if you want to appent the demographic data into density file, you can do that to make it easier for further analysis

In [ ]:
# ignore the following code, the following code is trying to extract the control data

import os
import pandas as pd
import numpy as np

# ----- CONFIG -----
input_dir = "/Users/linyufen/Downloads/uwa_filtered_0to1_csv"
uwa_control_codes = ["6672", "6774", "6789", "6805", "6815", "6904", "6992", "7065"]
out_code_stats = "/Users/linyufen/Downloads/uwa_control_0to1_stats.csv"       # per-code + overall
out_file_stats = "/Users/linyufen/Downloads/uwa_control_0to1_file_stats.csv"  # per-file
# -------------------

def pick_vol_column(df: pd.DataFrame) -> str:
    # Prefer an explicit *_vol column; otherwise fallback to the first column
    vol_cols = [c for c in df.columns if c.endswith("_vol")]
    return vol_cols[0] if vol_cols else df.columns[0]

def main():
    if not os.path.isdir(input_dir):
        raise FileNotFoundError(f"Input folder not found: {input_dir}")

    all_values = []
    values_by_code = {code: [] for code in uwa_control_codes}
    file_stats = []

    for fname in os.listdir(input_dir):
        if not fname.endswith(".csv"):
            continue
        # only files starting with one of the control codes
        code_match = next((c for c in uwa_control_codes if fname.startswith(c)), None)
        if code_match is None:
            continue

        fpath = os.path.join(input_dir, fname)
        df = pd.read_csv(fpath)

        vol_col = pick_vol_column(df)
        vals = pd.to_numeric(df[vol_col], errors="coerce").dropna()

        # If your folder already contains 0..1 filtered values, no need to filter again.
        # If you want to enforce it here too, uncomment the next line:
        # vals = vals[(vals >= 0) & (vals <= 1)]

        if len(vals) == 0:
            # still record an entry with NaNs
            file_stats.append({
                "filename": fname,
                "code": code_match,
                "count": 0,
                "mean": np.nan,
                "std": np.nan,
            })
            continue

        all_values.extend(vals.tolist())
        values_by_code[code_match].extend(vals.tolist())

        file_stats.append({
            "filename": fname,
            "code": code_match,
            "count": int(len(vals)),
            "mean": float(vals.mean()),
            "std": float(vals.std(ddof=1)),  # sample std
        })

    # Per-code + overall stats
    rows = []
    for code, vals in values_by_code.items():
        if len(vals) > 0:
            s = pd.Series(vals)
            rows.append({
                "code": code,
                "count": int(len(s)),
                "mean": float(s.mean()),
                "std": float(s.std(ddof=1)),
            })
        else:
            rows.append({"code": code, "count": 0, "mean": np.nan, "std": np.nan})

    # Overall
    if len(all_values) > 0:
        s_all = pd.Series(all_values)
        rows.append({
            "code": "OVERALL",
            "count": int(len(s_all)),
            "mean": float(s_all.mean()),
            "std": float(s_all.std(ddof=1)),
        })
    else:
        rows.append({"code": "OVERALL", "count": 0, "mean": np.nan, "std": np.nan})

    code_stats_df = pd.DataFrame(rows)
    code_stats_df.to_csv(out_code_stats, index=False)

    file_stats_df = pd.DataFrame(file_stats).sort_values(["code", "filename"])
    file_stats_df.to_csv(out_file_stats, index=False)

    print(f"✅ Saved per-code stats to: {out_code_stats}")
    print(f"✅ Saved per-file stats to: {out_file_stats}")

if __name__ == "__main__":
    main()



# ----- CONFIG -----
input_dir = "/Users/linyufen/Downloads/uwa_filtered_1to15_csv"
uwa_control_codes = ["6672", "6774", "6789", "6805", "6815", "6904", "6992", "7065"]
out_code_stats = "/Users/linyufen/Downloads/uwa_control_1to15_stats.csv"       # per-code + overall
out_file_stats = "/Users/linyufen/Downloads/uwa_control_1to15_file_stats.csv"  # per-file
# -------------------

def pick_vol_column(df: pd.DataFrame) -> str:
    # Prefer an explicit *_vol column; otherwise fallback to the first column
    vol_cols = [c for c in df.columns if c.endswith("_vol")]
    return vol_cols[0] if vol_cols else df.columns[0]

def main():
    if not os.path.isdir(input_dir):
        raise FileNotFoundError(f"Input folder not found: {input_dir}")

    all_values = []
    values_by_code = {code: [] for code in uwa_control_codes}
    file_stats = []

    for fname in os.listdir(input_dir):
        if not fname.endswith(".csv"):
            continue
        # only files starting with one of the control codes
        code_match = next((c for c in uwa_control_codes if fname.startswith(c)), None)
        if code_match is None:
            continue

        fpath = os.path.join(input_dir, fname)
        df = pd.read_csv(fpath)

        vol_col = pick_vol_column(df)
        vals = pd.to_numeric(df[vol_col], errors="coerce").dropna()

        # If your folder already contains 0..1 filtered values, no need to filter again.
        # If you want to enforce it here too, uncomment the next line:
        # vals = vals[(vals >= 0) & (vals <= 1)]

        if len(vals) == 0:
            # still record an entry with NaNs
            file_stats.append({
                "filename": fname,
                "code": code_match,
                "count": 0,
                "mean": np.nan,
                "std": np.nan,
            })
            continue

        all_values.extend(vals.tolist())
        values_by_code[code_match].extend(vals.tolist())

        file_stats.append({
            "filename": fname,
            "code": code_match,
            "count": int(len(vals)),
            "mean": float(vals.mean()),
            "std": float(vals.std(ddof=1)),  # sample std
        })

    # Per-code + overall stats
    rows = []
    for code, vals in values_by_code.items():
        if len(vals) > 0:
            s = pd.Series(vals)
            rows.append({
                "code": code,
                "count": int(len(s)),
                "mean": float(s.mean()),
                "std": float(s.std(ddof=1)),
            })
        else:
            rows.append({"code": code, "count": 0, "mean": np.nan, "std": np.nan})

    # Overall
    if len(all_values) > 0:
        s_all = pd.Series(all_values)
        rows.append({
            "code": "OVERALL",
            "count": int(len(s_all)),
            "mean": float(s_all.mean()),
            "std": float(s_all.std(ddof=1)),
        })
    else:
        rows.append({"code": "OVERALL", "count": 0, "mean": np.nan, "std": np.nan})

    code_stats_df = pd.DataFrame(rows)
    code_stats_df.to_csv(out_code_stats, index=False)

    file_stats_df = pd.DataFrame(file_stats).sort_values(["code", "filename"])
    file_stats_df.to_csv(out_file_stats, index=False)

    print(f"✅ Saved per-code stats to: {out_code_stats}")
    print(f"✅ Saved per-file stats to: {out_file_stats}")

if __name__ == "__main__":
    main()

✅ Saved per-code stats to: /Users/linyufen/Downloads/uwa_control_0to1_stats.csv
✅ Saved per-file stats to: /Users/linyufen/Downloads/uwa_control_0to1_file_stats.csv
✅ Saved per-code stats to: /Users/linyufen/Downloads/uwa_control_1to15_stats.csv
✅ Saved per-file stats to: /Users/linyufen/Downloads/uwa_control_1to15_file_stats.csv


In [31]:
# file_sd_summary.py
import os
import pandas as pd
import numpy as np

# ---- CONFIG ----
input_dir = "/Users/linyufen/Downloads/uwa_filtered_0to1_csv"
output_csv = "/Users/linyufen/Downloads/uwa_file_0to1_sd_summary.csv"

GLOBAL_MEAN = 0.170681424
GLOBAL_STD  = 0.237833299
# ----------------

T1 = GLOBAL_MEAN + 1 * GLOBAL_STD
T2 = GLOBAL_MEAN + 2 * GLOBAL_STD
T3 = GLOBAL_MEAN + 3 * GLOBAL_STD

def pick_vol_column(df: pd.DataFrame) -> str:
    # Prefer an explicit *_vol column; otherwise fall back to the first column
    vol_cols = [c for c in df.columns if c.endswith("_vol")]
    return vol_cols[0] if vol_cols else df.columns[0]

rows = []
for fname in os.listdir(input_dir):
    if not fname.endswith(".csv"):
        continue

    fpath = os.path.join(input_dir, fname)
    df = pd.read_csv(fpath)

    vol_col = pick_vol_column(df)
    vals = pd.to_numeric(df[vol_col], errors="coerce").dropna()

    total = len(vals)
    file_mean = float(vals.mean()) if total else np.nan

    def above_stats(thresh):
        sel = vals[vals > thresh]
        n = len(sel)
        mean_sel = float(sel.mean()) if n else np.nan
        prop_sel = (n / total) if total else np.nan
        return mean_sel, prop_sel

    mean_above_1sd, prop_above_1sd = above_stats(T1)
    mean_above_2sd, prop_above_2sd = above_stats(T2)
    mean_above_3sd, prop_above_3sd = above_stats(T3)

    rows.append({
        "filename": fname,
        "code": fname.split("_", 1)[0],
        "count": total,
        "file_mean": file_mean,
        "mean_above_1SD": mean_above_1sd,
        "mean_above_2SD": mean_above_2sd,
        "mean_above_3SD": mean_above_3sd,
        "prop_above_1SD": prop_above_1sd,
        "prop_above_2SD": prop_above_2sd,
        "prop_above_3SD": prop_above_3sd,
    })

pd.DataFrame(rows).sort_values(["code", "filename"]).to_csv(output_csv, index=False)
print(f"✅ Saved: {output_csv}")
print(f"Thresholds -> T1: {T1:.6f}, T2: {T2:.6f}, T3: {T3:.6f}")



# ---- CONFIG ----
input_dir = "/Users/linyufen/Downloads/uwa_filtered_1to15_csv"
output_csv = "/Users/linyufen/Downloads/uwa_file_1to15_sd_summary.csv"

GLOBAL_MEAN = 0.170681424	
GLOBAL_STD  = 0.237833299
# ----------------

T1 = GLOBAL_MEAN + 1 * GLOBAL_STD
T2 = GLOBAL_MEAN + 2 * GLOBAL_STD
T3 = GLOBAL_MEAN + 3 * GLOBAL_STD

def pick_vol_column(df: pd.DataFrame) -> str:
    # Prefer an explicit *_vol column; otherwise fall back to the first column
    vol_cols = [c for c in df.columns if c.endswith("_vol")]
    return vol_cols[0] if vol_cols else df.columns[0]

rows = []
for fname in os.listdir(input_dir):
    if not fname.endswith(".csv"):
        continue

    fpath = os.path.join(input_dir, fname)
    df = pd.read_csv(fpath)

    vol_col = pick_vol_column(df)
    vals = pd.to_numeric(df[vol_col], errors="coerce").dropna()

    total = len(vals)
    file_mean = float(vals.mean()) if total else np.nan

    def above_stats(thresh):
        sel = vals[vals > thresh]
        n = len(sel)
        mean_sel = float(sel.mean()) if n else np.nan
        prop_sel = (n / total) if total else np.nan
        return mean_sel, prop_sel

    mean_above_1sd, prop_above_1sd = above_stats(T1)
    mean_above_2sd, prop_above_2sd = above_stats(T2)
    mean_above_3sd, prop_above_3sd = above_stats(T3)

    rows.append({
        "filename": fname,
        "code": fname.split("_", 1)[0],
        "count": total,
        "file_mean": file_mean,
        "mean_above_1SD": mean_above_1sd,
        "mean_above_2SD": mean_above_2sd,
        "mean_above_3SD": mean_above_3sd,
        "prop_above_1SD": prop_above_1sd,
        "prop_above_2SD": prop_above_2sd,
        "prop_above_3SD": prop_above_3sd,
    })

pd.DataFrame(rows).sort_values(["code", "filename"]).to_csv(output_csv, index=False)
print(f"✅ Saved: {output_csv}")
print(f"Thresholds -> T1: {T1:.6f}, T2: {T2:.6f}, T3: {T3:.6f}")


✅ Saved: /Users/linyufen/Downloads/uwa_file_0to1_sd_summary.csv
Thresholds -> T1: 0.408515, T2: 0.646348, T3: 0.884181
✅ Saved: /Users/linyufen/Downloads/uwa_file_1to15_sd_summary.csv
Thresholds -> T1: 0.408515, T2: 0.646348, T3: 0.884181
